This notebook incrementally calculates profit estimates of performing skill based actions on an hourly basis and writes the data to  
Table xxx

In [ ]:
import pyspark.sql.functions as sf
from time import time
from delta.tables import DeltaTable

In [ ]:
catalog = dbutils.widgets.get("catalog")

Enriched Data

In [ ]:
# Read data from 'runescape.02_silver.1h_prices_enriched'
df_prices = spark.read.table(f"{catalog}.02_silver.1h_prices_enriched")
df_prices = df_prices.drop("highalch","limit","members")

In [ ]:
# get skill input data
df_skill_inputs = spark.read.table(f"{catalog}.02_silver.skill_action_inputs_enriched")

In [ ]:
# Read data from 'runescape.02_silver.1h_prices_last_enriched'
# TODO figure out a better name for this df...
df_prices_last = spark.read.table(f"{catalog}.02_silver.1h_prices_last_enriched")
# drop time column
df_prices_last = df_prices_last.drop("time", "highalch", "limit", "members")

In [ ]:
# TODO replace the code below with code that gets the most recent data from 'runescape.02_silver.1h_prices_enriched'
# instead of using the time window
# will need to make sure that the merge works when writing to gold layer.

In [ ]:
# TODO implement gold layer check to ensure that data does not already exist for this time.

In [ ]:
# Get the most recent time data inserted into 'runescape.02_silver.1h_prices_enriched'
max_time = df_prices.select(sf.max("time")).collect()[0][0]
print(max_time)
df_prices_latest = df_prices.filter(f"time = {max_time}")

In [ ]:
df_prices_latest.show()

Need to create a dataframe for all skill action outputs (and prices/ volumes) and thier respective inputs (and prices/ volumes)  
This dataframe also should have price data (and inputs) for items that did not have data in the last hour and instead use historic data from table 1h_prices_last_enriched

In [ ]:
# get df with all distinct skill action output and input id's
# TODO consider how this is impacted by implementation of other skills
# TODO consider renaming df_outputs_time to something less dumb
df_outputs_time = df_skill_inputs\
    .select("id_output")\
    .union(df_skill_inputs\
    .select("id_input"))\
    .dropDuplicates()\
    .withColumnRenamed("id_output","id")\
    .withColumn('time',sf.lit(f"{max_time}"))


In [ ]:
df_outputs_time.show()

Data imputation for missing/ filtered out data.  
Imputation method: Use the last known avg values based on hourly data in table 'runescape.02_silver.1h_prices_last_enriched'  
This approach has it's limiations and should be reconsidered.

In [ ]:
# create df with imputed price data using the prices from 1h_prices_last_enriched
# This data will be used when there is no data for the respective item for profit calcs
df_prices_imputed = df_prices_last.join(df_outputs_time, "id")

In [ ]:
# create df with only items that have missing 'real' price data
df_prices_missing = df_prices_imputed.join(df_prices_latest.select("id"),"id","left_anti")

In [ ]:
df_prices_missing.sort("id").show()

In [ ]:
# union price data to get complete data set
df_prices_full = df_prices_missing.unionByName(df_prices_latest)

Build table with profit calculations

In [ ]:
# create df with full price data set combined with inputs (qty, id's)
# this will be used to calculate the input cost for each skill action
# inner join to only get data that has skill action
# TODO make sure this shouldn't be a left_outer join
df_prices_full_w_inputs = df_prices_full.join(
    df_skill_inputs, df_prices_full.id == df_skill_inputs.id_output, "inner"
)
df_prices_full_w_inputs = df_prices_full_w_inputs.drop(
    "id", "name", "highalch", "limit", "members"
)
# rename columns to be associated with output data
df_prices_full_w_inputs = df_prices_full_w_inputs.withColumnsRenamed(
    {
        "avg1HourHigh": "output_avg_high",
        "avg1HourHighVolume": "output_high_volume",
        "avg1HourLow": "output_avg_low",
        "avg1HourLowVolume": "output_low_volume",
    }
)


In [ ]:
df_prices_full_w_inputs.show()

In [ ]:
# create final price df with input costs
df_prices_final = df_prices_full_w_inputs.join(
    df_prices_full,
    [
        df_prices_full_w_inputs.id_input == df_prices_full.id,
        df_prices_full_w_inputs.time == df_prices_full.time,
    ],
    "left_outer",
).drop(df_prices_full_w_inputs.time, "id", "name")\
.withColumnsRenamed(
    {
        "avg1HourHigh": "input_avg_high",
        "avg1HourHighVolume": "input_high_volume",
        "avg1HourLow": "input_avg_low",
        "avg1HourLowVolume": "input_low_volume",
    }
)


In [ ]:
df_prices_final.show()

In [ ]:
# calculate input costs
df_input_cost = df_prices_final.withColumns(
    {
        "input_cost_low": sf.round(sf.col("input_avg_low") * sf.col("input_qty"), 1),
        "input_cost_high": sf.round(sf.col("input_avg_high") * sf.col("input_qty"), 1),
    }
)


In [ ]:
df_input_cost.show()

In [ ]:
# Summarize total input costs for each output
df_input_total_cost = df_input_cost.groupBy("time", "output", "skill").sum(
    "input_cost_high", "input_cost_low"
)
df_input_total_cost = df_input_total_cost.withColumnsRenamed(
    {
        "sum(input_cost_high)": "total_input_cost_high",
        "sum(input_cost_low)": "total_input_cost_low",
    }
)


In [ ]:
df_input_total_cost.show()

Calculate profit estimates based on input costs and GE Tax  
GE tax needs to be replaced with the GE tax function, but is close enough for now.

In [ ]:
# calculate profits
# 1 profit estimate record (high and low) calculated for a given output, skill combination
# TODO add tax and implement as function
tax = 0.98  # simplified tax for now, can be implemented as function
df_profits = (
    df_prices_final\
        .select("time","output","skill","output_avg_high","output_high_volume","output_avg_low","output_low_volume","id_output")\
        .dropDuplicates(["output", "skill"])\
        .join(df_input_total_cost, ["time", "output", "skill"])\
    .withColumns(
        {
            "profit_low_est": sf.round(
                tax * sf.col("output_avg_low") - sf.col("total_input_cost_high"), 2
            ),
            "profit_high_est": sf.round(
                tax * sf.col("output_avg_high") - sf.col("total_input_cost_low"), 2
            ),
        }
    )
)


In [ ]:
df_profits.show()

Write final profit calculation data to gold layer unity catalog table

In [ ]:
# save data to gold layer Unity Catalog table 'runescape.03_gold.1h_skill_profit'
targetDF = DeltaTable.forName(spark, f"{catalog}.03_gold.1h_skill_profit")
dfUpdates = df_profits
# Insert only data that is not already in target table
(targetDF.alias("t")
  .merge(
    source = dfUpdates.alias("s"),
    condition = "t.time = s.time AND t.id_output = s.id_output AND t.skill = s.skill")
  .whenNotMatchedInsertAll()
  .execute())